In [2]:
import os
import shutil
import yaml
from tqdm import tqdm  # Import tqdm for progress bars

# --- CONFIGURATION ---
SOURCE_DIR = r'D:\Backup\WORK\MACH-3D\DefectClassification\Cleaned_dataset'
TARGET_DIR = r'D:\Backup\WORK\MACH-3D\DefectClassification\Cleaned_dataset_OBB'

# Splits to process
splits = ['train', 'valid', 'test']

def convert_hbb_to_obb(source_path, target_path):
    """Converts [cls, xc, yc, w, h] to [cls, x1, y1, x2, y2, x3, y3, x4, y4]"""
    with open(source_path, 'r') as f:
        lines = f.readlines()

    obb_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        
        cls, xc, yc, w, h = map(float, parts)
        
        # Calculate 4 corners (unrotated)
        x1, y1 = xc - w/2, yc - h/2
        x2, y2 = xc + w/2, yc - h/2
        x3, y3 = xc + w/2, yc + h/2
        x4, y4 = xc - w/2, yc + h/2
        
        # Format: class x1 y1 x2 y2 x3 y3 x4 y4
        obb_line = f"{int(cls)} {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}\n"
        obb_lines.append(obb_line)

    with open(target_path, 'w') as f:
        f.writelines(obb_lines)

# 1. Create directory structure
print("Creating directory structure...")
for split in splits:
    os.makedirs(os.path.join(TARGET_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(TARGET_DIR, split, 'labels'), exist_ok=True)

# 2. Process Files
print(f"Starting conversion from {SOURCE_DIR} to {TARGET_DIR}...")

for split in splits:
    img_dir = os.path.join(SOURCE_DIR, split, 'images')
    lbl_dir = os.path.join(SOURCE_DIR, split, 'labels')
    
    if not os.path.exists(lbl_dir):
        print(f"Warning: Labels folder not found for '{split}', skipping...")
        continue

    # Get list of text files first to feed into tqdm
    label_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    
    # Wrap the loop with tqdm for the progress bar
    print(f"Processing '{split}' set...")
    for label_file in tqdm(label_files, desc=f"{split.upper()}", unit="file"):
        # Convert Label
        convert_hbb_to_obb(
            os.path.join(lbl_dir, label_file),
            os.path.join(TARGET_DIR, split, 'labels', label_file)
        )
        
        # Copy corresponding image
        img_name_base = os.path.splitext(label_file)[0]
        # Check for common extensions
        for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            img_path = os.path.join(img_dir, img_name_base + ext)
            if os.path.exists(img_path):
                shutil.copy(img_path, os.path.join(TARGET_DIR, split, 'images', img_name_base + ext))
                break

# 3. Create New YAML
print("\nGenerating data_obb.yaml...")
data_yaml = {
    'path': TARGET_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {
        0: 'blobs', 1: 'cracks', 2: 'over_extrusion', 3: 'spaghetti',
        4: 'stringing', 5: 'under_extrusion', 6: 'layer_shift', 7: 'warp'
    },
    'nc': 8
}

yaml_path = os.path.join(TARGET_DIR, 'data_obb.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"Done! OBB dataset ready at: {yaml_path}")

Creating directory structure...
Starting conversion from D:\Backup\WORK\MACH-3D\DefectClassification\Cleaned_dataset to D:\Backup\WORK\MACH-3D\DefectClassification\Cleaned_dataset_OBB...
Processing 'train' set...


TRAIN: 100%|██████████| 25972/25972 [05:38<00:00, 76.73file/s]


Processing 'valid' set...


VALID: 100%|██████████| 7420/7420 [02:28<00:00, 49.86file/s]


Processing 'test' set...


TEST: 100%|██████████| 3711/3711 [01:10<00:00, 52.35file/s]


Generating data_obb.yaml...
Done! OBB dataset ready at: D:\Backup\WORK\MACH-3D\DefectClassification\Cleaned_dataset_OBB\data_obb.yaml
